# 02 — XGBoost 단일 회귀

Anchor ±50% Narrow HPO + 후처리 매트릭스. (1차 reg_only 데이터 없음 — Y>0 우회 anchor 사용)

- **입력**: `0_data/compet_xs_data.csv`, `compet_ys_*_data.csv`
- **출력**: `4_output/02_reg_single/xgb/{best_params.json, fold_models.pkl, optuna_*.db, oof|val|test_die.csv, oof|val|test_unit.csv}`
- **PP**: 트리 공통 `PP_FIXED` ([strategy_common.md §1](../../strategy_common.md))
- **HPO**: 150 trial, anchor 첫 trial enqueue ([strategy.md §5.5](../strategy.md), Y>0 컨텍스트에서 fork — reg_only 1차 산출물 없음), narrow ±50% ([§7.5](../strategy.md))
- **손실함수**: `reg:squarederror / count:poisson / reg:tweedie` 3종 ([§6](../strategy.md))
- **target transform**: `'none'` 고정 (strategy_common §24 — log1p_check 검증)

## 모듈 의존성 ([strategy.md §13](../strategy.md))

1. `3_modeling/modules/` 이관 완료
2. `models.py` — `xgb_space`에 `count:poisson` 추가, `reg:tweedie_*` categorical → `reg:tweedie` + `tweedie_variance_power=suggest_float(1.05, 1.95)` 통합
3. `hpo.py` — `enqueue_trials` 인자 + trial 내 target_transform 분기 (`reg:tweedie` 시 OFF)
4. `postprocess.py` — `Q25/Q75` + `zero_clip_space='log'` 분기

## 결정 필요 (strategy.md §14)



## 1. 환경 설정 + 모듈 import

In [ ]:
import os, sys

# Google Drive 파일 ID들 — Colab에서 코드/데이터/모듈 zip을 자동으로 받아 풀 때 사용 (로컬은 무시)
GDRIVE_CODE_ID          = '1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I'  # code.zip = setup.py + utils/
GDRIVE_DATASET_ID       = '1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO'  # dataset.zip = 원본 CSV 4개
GDRIVE_PREPROCESSING_ID = '1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr'  # preprocessing.zip = cleaning/outlier/scaling 등
GDRIVE_MODELING_ID      = '1Vrn5LBl611rWbag7d09LZH68_lfpu6wP'   # modeling.zip = 3_modeling/modules (코드 수정 시 재업로드)
GDRIVE_OUTPUT_ID       = '1ts73qEMmjX8cKIb-QeDQ-TMeyudFGWzs'  # 4_output.zip = 기존 실험 산출물 (RESUME용)
RESUME                 = True   # True=기존 optuna db에 trial 이어 붙임 / False=처음부터 (db 있으면 의도적 에러)

# Colab이면 필요한 zip들을 받아 풀고(이미 풀려 있으면 skip), 로컬이면 ../../../setup.py만
try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system(f'gdown {GDRIVE_CODE_ID} -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system(f'gdown {GDRIVE_DATASET_ID} -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system(f'gdown {GDRIVE_PREPROCESSING_ID} -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    if not os.path.exists('/content/project/3_modeling/modules/hpo.py'):
        assert GDRIVE_MODELING_ID, 'GDRIVE_MODELING_ID가 비어있음 — modules.zip Drive ID 입력 필요'
        os.makedirs('/content/project/3_modeling', exist_ok=True)
        os.system(f'gdown {GDRIVE_MODELING_ID} -O /content/modules.zip')
        os.system('unzip -qo /content/modules.zip -d /content/project/3_modeling')
    if RESUME and GDRIVE_OUTPUT_ID and not os.path.exists('/content/project/4_output/01_zit'):
        os.system(f'gdown {GDRIVE_OUTPUT_ID} -O /content/4_output.zip')
        os.system('unzip -qo /content/4_output.zip -d /content/project')
        os.remove('/content/4_output.zip')
        # 모듈 경로 등록 — Colab 런타임은 세션마다 sys.path가 초기화됨
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# 공통 유틸: 경로 상수(OUTPUT_DIR, DATA_DIR), 컬럼 상수(TARGET_COL, KEY_COL), SEED
from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs

# 전처리 모듈(2_preprocessing) 경로 + `from modules import ...` 가 3_modeling/modules를 찾게
# 전처리 모듈(2_preprocessing/)과 모델링 모듈(3_modeling/modules/) 모두 sys.path 등록
# 노트북 위치가 달라도 PROJECT_ROOT 기준 절대경로로 접근 → Colab·로컬 동일
PREP_ROOT = os.path.join(PROJECT_ROOT, '2_preprocessing')
if PREP_ROOT not in sys.path:
    sys.path.insert(0, PREP_ROOT)
MODEL_ROOT = os.path.join(PROJECT_ROOT, '3_modeling')
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

# preprocess.run: 전체 전처리 파이프라인(결측/이상치/스케일/집계)
# hpo: HPO(run_hpo) + 재학습(refit_best) + 산출물 저장(save_artifacts)
# models: 모델 레지스트리 (AVAILABLE_MODELS 리스트 + 모델 생성 팩토리)
from modules import preprocess, hpo, models   # noqa: E402  (preprocess.run, hpo.run_hpo/refit_best/save_artifacts, models 레지스트리)
# meta_features: position·die_xy 메타피처 생성 (run_wf_xy 파싱 기반)
from meta_features import add_meta_features

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

## 2. 실험 설정

In [ ]:
# 모델 고정 (이 노트북은 XGBoost 단일)
MODEL_NAME = 'xgb'
EXP_ID     = f'reg-{MODEL_NAME}-002'
EXP_MEMO   = 'Anchor ±50% Narrow (Y>0 우회 base, 1차 reg_only 데이터 없음)'
USER       = 'jh'

# Optuna 예산 (XGBoost: LGBM과 동일 3000 trial)
N_TRIALS = 3000
N_FOLDS  = 5
# N_STARTUP_TRIALS: 랜덤 탐색 후 TPE 전환 — 초반 공간 커버리지 확보
N_STARTUP_TRIALS = 50
N_JOBS   = -1   # 모델 학습 병렬도 (-1 = 전체 코어)
TIMEOUT_SEC      = 20 * 60 * 60  # 초 단위, None=무제한

# TARGET_TRANSFORM='none': 트리는 log1p 등 target 변환이 RMSE에 유의미한 차이 없음
TARGET_TRANSFORM = 'none'  # 트리는 'none' 통일
CLIP_Y_EXTREME   = True

# 출력 경로 — EXP_ID 끝자리('002')를 하위 폴더명으로 사용
OUT_DIR = os.path.join(OUTPUT_DIR, '02_reg_single', MODEL_NAME, 'hp', EXP_ID.split('-')[-1])
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# 트리 공통 전처리 고정 파라미터 — strategy_common.md §1
PP_FIXED = {
    'missing_threshold':          0.30,
    'corr_threshold':             0.90,
    'corr_keep_by':               'std',
    'add_indicator':              True,
    'indicator_threshold':        0.05,
    'spatial_max_dist':           6.0,
    'post_impute_corr_threshold': 0.96,
    'post_impute_corr_keep_by':   'std',
}

# anchor — Y>0 컨텍스트에서 우회로 얻은 base HP (1차 reg_only 데이터가 없어서). objective는 categorical 후보 중 하나
XGB_ANCHOR = {
        # reg:tweedie: compound Poisson-Gamma — 0 확률질량 + 양수 연속분포 (health 구조 일치)
    'objective':              'reg:tweedie',  # 후보: reg:squarederror | count:poisson | reg:tweedie
        # tweedie_variance_power: 1.0=Poisson, 2.0=Gamma, 1.5=중간 (0 비율 71% 고려)
    'tweedie_variance_power': 1.5,
    'n_estimators':           1423,
    'learning_rate':          0.0363,
    'max_depth':              10,
    'min_child_weight':       0.621,
        # 서브샘플링 + 정규화: subsample/colsample_bytree + L1/L2/gamma(분기 최소 gain)
    'subsample':              0.728,
    'colsample_bytree':       0.618,
    'reg_alpha':              0.01680,
    'reg_lambda':             3.890e-06,
    'gamma':                  3.837e-06,
}

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}, N_JOBS={N_JOBS}, TIMEOUT_SEC={TIMEOUT_SEC}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'OUT_DIR={OUT_DIR}')

## 3. 데이터 로드 + target clip + transform

In [ ]:
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
# split_xs: xs['split'] 컬럼 기준으로 train/val/test 행 분리
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# train y의 극단값(1.0, 1건)만 두 번째로 큰 값으로 clip — 학습 입력 안정화 (원본 ys는 보존)
# ys는 원본 보존 — clip/transform은 ys_input 복사본에만 적용 (누수 방지)
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# 트리 모델은 target 변환이 결과를 거의 안 바꿔서 'none'으로 고정 (transform 함수 둘 다 None)
# 트리는 target 변환이 RMSE를 유의하게 낮추지 않음 (EDA max|r|=0.037 약신호)
target_transform_fn = None
target_inverse_fn   = None
print(f'[target transform] {TARGET_TRANSFORM} (strategy_common §24 — 트리 target_transform=none 통일)')

## 4. 전처리 (PP_FIXED 고정)

In [ ]:
# PP_FIXED: missing_threshold(결측 제거 기준), corr_threshold(다중공선성), add_indicator(결측 indicator 추가)
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PP_FIXED)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']

# 메타피처: 트리는 position을 raw 정수로, die_x/die_y를 연속형으로 추가
# position_mode='raw': 위치를 1~4 정수로 직접 피처 추가 (범주형 OHE 없음)
feat_cols_clean = add_meta_features(
    xs_train, xs_val, xs_test, feat_cols_clean,
    position_mode='raw', use_die_xy=True,
)

# fit은 train에서만, transform은 train/val/test 모두 — 데이터 누수 방지
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')

## 5. Optuna HPO (anchor 첫 trial enqueue + narrow ±50%)

[strategy.md §7.5](../strategy.md): 1차 reg_only 데이터 없어 anchor ±50% narrow. xgb_space는 §13 #2 변경 후 `count:poisson` 후보를 포함해야 함.

In [ ]:
# study_meta: HPO 실험 메타정보를 DB(user_attr)에 박제 → trial별 EXP_ID/anchor/PP 설정 조회 가능
study_meta_for_save = {
    'exp_id':              EXP_ID,
    'exp_memo':            EXP_MEMO,
    'user':                USER,
    'model_name':          MODEL_NAME,
    'target_transform':    TARGET_TRANSFORM,
    'clip_y_extreme':      CLIP_Y_EXTREME,
        # effective_pp_params: preprocess.run이 실제로 사용한 파라미터 (FIXED 기본값 + override 반영)
    'effective_pp_params': pp['effective_params'],
    'n_trials':            N_TRIALS,
    'n_folds':             N_FOLDS,
    'n_jobs':              N_JOBS,
        # n_startup_trials: TPE 이전 랜덤 탐색 횟수 — 공간 초기 커버리지 확보
    'n_startup_trials':    N_STARTUP_TRIALS,
    'timeout_sec':         TIMEOUT_SEC,
    'seed_kfold':          SEED,
    'anchor':              XGB_ANCHOR,
}

# anchor를 trial 0으로 강제하려면 study를 먼저 만들어 enqueue → 그 다음 run_hpo가 같은 study(이름·storage)에 이어서
# Optuna study 설정 — storage=SQLite로 trial 결과 영구 저장 (Colab 재시작 후 RESUME 가능)
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
# direction='minimize': OOF RMSE 최소화. sampler·pruner 설정은 study 생성 시 1회만
_study = optuna.create_study(
    direction='minimize',
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    load_if_exists=RESUME,
        # TPESampler: multivariate=True → 파라미터 간 상관 모델링. seed=None → 실행마다 다른 탐색 경로
    sampler=TPESampler(seed=None, multivariate=True, group=True, n_startup_trials=N_STARTUP_TRIALS),
        # MedianPruner: n_warmup=10 trial 이후 중간값 미달 trial 조기 중단 → 탐색 효율 향상
    pruner=MedianPruner(n_warmup_steps=10),
)
# enqueue: 기존 trial이 없을 때만 anchor를 trial 0으로 강제 → RESUME 시 중복 방지
hpo.enqueue_anchor(_study, XGB_ANCHOR)   # 기존 trial 없을 때만 enqueue (RESUME 시 중복 방지)

# HPO 본체: unit 단위 KFold OOF → unit RMSE를 minimize. val/test RMSE는 매 trial user_attr에 기록
res = hpo.run_hpo(
    xs_train=xs_train,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    n_trials=N_TRIALS,
    n_folds=N_FOLDS,
        # n_jobs: 병렬 fold 학습 (-1=전체 코어). N_FOLDS=5이므로 최대 5 worker가 동시 학습
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
    study_name=EXP_ID,
    storage=f'sqlite:///{DB_PATH}',
    resume_study=RESUME,
    timeout=TIMEOUT_SEC,
    user_attrs=study_meta_for_save,
    xs_val=xs_val,   ys_val_unit=ys_input['validation'],
    xs_test=xs_test, ys_test_unit=ys_input['test'],
)
# res에서 best trial 정보 추출 → 재학습(다음 cell)에 사용
study                 = res['study']
best_params_for_refit = res['best_params']
# best_value: HPO 목적함수(OOF RMSE) 최솟값 → study_meta에 저장하여 산출물과 함께 기록
study_meta_for_save['hpo_best_value'] = float(res['best_value'])

# trial 0이 anchor 키들을 그대로 갖고 있는지 확인 (enqueue 정상 동작)
first_trial_params = study.trials[0].params
anchor_keys_present = {k: first_trial_params.get(k) for k in XGB_ANCHOR if k in first_trial_params}
print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
print(f'[검증] trial 0 params (anchor 키만): {anchor_keys_present}')
print(f'best_params = {best_params_for_refit}')

## 6. Best trial 재학습 (K-fold OOF)

In [ ]:
# best HP로 5-fold 재학습 → die-level OOF / val / test 예측 (val·test는 fold 평균)
# refit: HPO best_params로 N_FOLDS번 재학습 → die·unit 레벨 OOF/val/test 예측 생성
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params_for_refit,
    n_folds=N_FOLDS,
    n_jobs=N_JOBS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

# 후처리 이전(mean 집계) unit RMSE를 정답과 정렬해 계산 — train(OOF) / val / test
# die-level 예측을 unit-level mean 집계 후 RMSE 계산 (후처리 최적화 이전 baseline)
y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))

# val·test는 fold 평균 예측 (각 fold 모델이 동일 val/test에 predict → 평균)
y_val_true  = ys_input['validation'].set_index(KEY_COL)[TARGET_COL]
val_u       = final['val_pred_unit'].set_index(KEY_COL)['pred'].loc[y_val_true.index]
val_rmse    = float(np.sqrt(np.mean((val_u.values - y_val_true.values)**2)))

y_test_true = ys_input['test'].set_index(KEY_COL)[TARGET_COL]
test_u      = final['test_pred_unit'].set_index(KEY_COL)['pred'].loc[y_test_true.index]
test_rmse   = float(np.sqrt(np.mean((test_u.values - y_test_true.values)**2)))

# segment 분해: train y max=1.0 / val y max≈0.17 → RMSE 스케일 차이가 크므로 세트별 함께 확인
print(f'\n[Refit 완료] (original space, postprocess 이전)')
print(f'  OOF  unit RMSE = {oof_rmse:.6f}')
print(f'  val  unit RMSE = {val_rmse:.6f}')
print(f'  test unit RMSE = {test_rmse:.6f}')

## 7. 후처리 매트릭스 + 산출물 저장

In [ ]:
# 후처리 설정 — die→unit 집계 8종 중 best + zero_clip 임계값 탐색. trees는 log space 안 씀(target_transform='none'), π threshold 없음
# 후처리: die→unit 집계 방식 8종 × zero_clip 임계값 그리드 탐색 → val RMSE 최소 조합 선택
POSTPROCESS_CONFIG = {
    'agg_methods':      ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted', 'Q25', 'Q75'),
        # zero_clip: 임계값 이하 예측을 0으로 snap — False Positive(소량 예측) 제거
    'zero_clip_range':  (0.001, 0.015),
    'zero_clip_step':   0.001,
    'zero_clip_log_space': TARGET_TRANSFORM == 'log1p',
    'use_pi_threshold': False,
}

# fold_models.pkl + best_params.json + die/unit CSV 6개 저장 (postprocess_config 주면 unit CSV는 튜닝값)
# save_artifacts: fold_models.pkl + best_params.json + die·unit CSV 6개 → stacking 재사용 가능
hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],
    y_test_unit=ys_input['test'],
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

# 저장된 파일 목록
# 저장 완료 후 파일 목록 출력 (파일명 + 크기) — 누락 파일 즉시 확인
for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

# Colab이면 산출물을 zip으로 묶어 로컬 PC로 다운로드
# Colab 환경이면 OUT_DIR을 zip으로 묶어 로컬 PC에 자동 다운로드
try:
    import google.colab
    from google.colab import files
    import shutil
    _zip_base = os.path.join('/content', f'{MODEL_NAME}_{EXP_ID}_outputs')
    _zip_path = shutil.make_archive(_zip_base, 'zip', OUT_DIR)
    print(f'[zip 생성] {_zip_path} ({os.path.getsize(_zip_path)/1024:.1f} KB)')
    try:
        files.download(_zip_path)
    except Exception as _e:
        from IPython.display import FileLink, display
        print(f'[files.download 실패: {_e}] 아래 링크 클릭')
        display(FileLink(_zip_path))
except ImportError:
    pass